In [ ]:
# add CUDA_VISIBLE_DEVICES=1 to run on GPU 1

! export CUDA_VISIBLE_DEVICES=0
! XLA_PYTHON_CLIENT_MEM_FRACTION=0.9
! XLA_PYTHON_CLIENT_PREALLOCATE=false
! XLA_PYTHON_CLIENT_ALLOCATOR=platform


import dataclasses

import jax

from openpi.models import model as _model
from openpi.policies import droid_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

/home/youliang/miniconda3/envs/openpi/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Policy inference

The following example shows how to create a policy from a checkpoint and run inference on a dummy example.

In [2]:
config = _config.get_config("pi0_fast_droid")
checkpoint_dir = download.maybe_download(
    "s3://openpi-assets/checkpoints/pi0_fast_droid"
    # "s3://openpi-assets/checkpoints/pi0_fast_base"
)

# Create a trained policy.
policy = _policy_config.create_trained_policy(config, checkpoint_dir)
print(policy._input_transform)

# Run inference on a dummy example. This example corresponds to observations produced by the DROID runtime.
example = droid_policy.make_droid_example()
print(example)
result = policy.infer(example)

# Delete the policy to free up memory.
del policy

print("Actions shape:", result["actions"].shape)

Some kwargs in processor config are unused and will not have any effect: vocab_size, min_token, time_horizon, action_dim, scale. 
Some kwargs in processor config are unused and will not have any effect: vocab_size, min_token, time_horizon, action_dim, scale. 


CompositeTransform(transforms=[InjectDefaultPrompt(prompt=None), DroidInputs(action_dim=8, model_type=<ModelType.PI0_FAST: 'pi0_fast'>), Normalize(norm_stats={'actions': NormStats(mean=array([-4.51024157e-03,  1.61404632e-02,  1.44883636e-03,  3.11631333e-02,
        1.04391526e-03, -2.07705610e-04,  4.28350419e-03,  4.46768843e-01]), std=array([0.15498388, 0.30615082, 0.151005  , 0.30140114, 0.22559904,
       0.24724309, 0.26929824, 0.44116568]), q01=array([-0.4596, -0.8008, -0.4412, -0.9304, -0.6368, -0.6356, -0.7592,
        0.    ]), q99=array([0.442 , 0.7756, 0.4528, 0.7992, 0.6404, 0.6752, 0.7316, 0.9998])), 'state': NormStats(mean=array([ 0.01133531,  0.27205018, -0.01088267, -2.01687728, -0.03630283,
        2.34795714,  0.09651095,  0.39627547]), std=array([0.31480195, 0.48860569, 0.27389642, 0.48531356, 0.52181067,
       0.45630267, 0.74456466, 0.40620682]), q01=array([-0.91823926, -0.85370122, -0.84060832, -2.77147968, -1.79171449,
        1.23216674, -1.99517394,  0.     

# Working with a live model


The following example shows how to create a live model from a checkpoint and compute training loss. First, we are going to demonstrate how to do it with fake data.


In [ ]:
config = _config.get_config("pi0_aloha_sim")

checkpoint_dir = download.maybe_download("s3://openpi-assets/checkpoints/pi0_aloha_sim")
key = jax.random.key(0)

# Create a model from the checkpoint.
model = config.model.load(_model.restore_params(checkpoint_dir / "params"))

# We can create fake observations and actions to test the model.
obs, act = config.model.fake_obs(), config.model.fake_act()

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)
print("Loss shape:", loss.shape)

Now, we are going to create a data loader and use a real batch of training data to compute the loss.

In [ ]:
# Reduce the batch size to reduce memory usage.
config = dataclasses.replace(config, batch_size=2)

# Load a single batch of data. This is the same data that will be used during training.
# NOTE: In order to make this example self-contained, we are skipping the normalization step
# since it requires the normalization statistics to be generated using `compute_norm_stats`.
loader = _data_loader.create_data_loader(config, num_batches=1, skip_norm_stats=True)
obs, act = next(iter(loader))

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)

# Delete the model to free up memory.
del model

print("Loss shape:", loss.shape)